In [1]:
import pandas as pd
from openpyxl import load_workbook
from openai import AzureOpenAI
from concurrent.futures import ThreadPoolExecutor

In [2]:
def sheet_to_df(ws):
    data = []
    rows = list(ws.rows)
    
    # 第一列當 header
    headers = [cell.value for cell in rows[0]]
    
    for row in rows[1:]:
        values = []
        for cell in row:
            val = cell.value
            
            # 確保是字串
            if isinstance(val, str):
                val = val.replace("_x000D_", "\n")  # 修正 Excel 換行
            values.append(val)
        
        data.append(values)
    
    df = pd.DataFrame(data, columns=headers)
    return df

In [3]:
def merge_text_columns(df, key_cols, text_cols):
    data = df.groupby(key_cols)[text_cols].agg(lambda x: "\n".join(x.dropna().astype(str).unique())).reset_index()
    return data

In [4]:
def clear_columns(df):
    df.columns = df.columns.str.strip()
    return df

In [5]:
def aggregate_text(df, key,col):
    df_group = df.groupby(key)[col].agg(lambda x: "\n".join(x.dropna().astype(str))).reset_index()
    return df_group

In [6]:
def truncate(text, max_len=4000):
    return text[:max_len] if isinstance(text, str) else ""

In [7]:
def build_text(row):
    text = f"""
    [Case ID]
    {row['病歷編號']}

    [主訴]
    {row['病人主訴']}

    [病史]
    {row['主要病史']}

    [體檢]
    {row['身體檢查']}

    [合併症]
    {row['合併症']}

    [檢驗與影像]
    {row['主要檢查結果']}
    {row.get('Finding / Impression', '')}

    [病理報告]
    {row['病理報告']}

    [DETAIL]
    {row.get('DETAIL', '')}

    [PROBLEM]
    {row.get('PROBLEM', '')}

    """
    return text

In [8]:
api_key = 'xxx'
endpoint = 'ooo'
deployment = 'gpt-4o'
api_version='2024-12-01-preview'

client = AzureOpenAI(
    api_key = api_key,
    azure_endpoint = endpoint,
    api_version = api_version
)

In [9]:
file_path = 'Fungus個案資料_20260325_481765AA.xlsx'
wb = load_workbook(file_path)

ws1 = wb['出院病摘']
ws2 = wb['檢查報告']
ws3 = wb['OPNote']
ws4 = wb['POMR']

df1 = sheet_to_df(ws1)
df2 = sheet_to_df(ws2)
df3 = sheet_to_df(ws3)
df5 = sheet_to_df(ws4)

In [10]:
df1.shape, df2.shape, df3.shape, df5.shape

((11, 18), (22, 7), (10, 9), (28, 28))

In [11]:
len(df1['病歷編號'].unique()), len(df2['病歷編號'].unique()), len(df3['病歷編號'].unique()), len(df5['病歷編號'].unique())

(10, 8, 10, 10)

In [29]:
# df2

In [13]:
# df2_g

In [14]:
# 把 df1同病患、同時間的病例合併
key_cols = ['病歷編號', '入院日期', '出院日期']
text_cols = ['病人主訴', '主要病史', '身體檢查', '合併症', '主要檢查結果', '病理報告']

df1 = merge_text_columns(df1, key_cols, text_cols)

In [15]:
# 把 df5同病患、同時間的病例合併
key_cols = ['病歷編號', '入院日期', '出院日期']
text_cols = ['PROBLEM', 'SDATA']

df5 = merge_text_columns(df5, key_cols, text_cols)

In [16]:
# 聚合其他table
df2_g = aggregate_text(df2, '病歷編號', 'Finding / Impression')
df3_g = aggregate_text(df3, '病歷編號', 'DETAIL') # 'OPERATION', 'PATHOLOGY'
df5_g = aggregate_text(df5, '病歷編號', 'PROBLEM')

In [17]:
base_cols = ['病歷編號', '病人主訴', '主要病史', '身體檢查', '合併症', '主要檢查結果', '病理報告']

df = df1[base_cols].copy()
df = df.merge(df3_g, on='病歷編號', how='left')
df = df.merge(df3[['病歷編號', 'OPERATION', 'PATHOLOGY']], on='病歷編號', how='left') # 併入 df3的某些欄位
df = df.merge(df5_g, on='病歷編號', how='left')
df = df.merge(df5[['病歷編號', 'SDATA']], on='病歷編號', how='left') # 併入 df5的SDATA
df = df.merge(df2_g, on='病歷編號', how='left')

In [18]:
df.columns

Index(['病歷編號', '病人主訴', '主要病史', '身體檢查', '合併症', '主要檢查結果', '病理報告', 'DETAIL',
       'OPERATION', 'PATHOLOGY', 'PROBLEM', 'SDATA', 'Finding / Impression'],
      dtype='object')

In [19]:
# 去掉Nan -> 空格
df_cols = ['病歷編號', '病人主訴', '主要病史', '身體檢查', '合併症', '主要檢查結果', '病理報告', 'DETAIL',
           'OPERATION', 'PATHOLOGY', 'PROBLEM', 'SDATA', 'Finding / Impression']

for col in df_cols:
    if col in df.columns:
        df[col] = df[col].fillna("")

In [20]:
# 限制文本長度

for col in df_cols:
    if col in df.columns:
        df[col] = df[col].apply(truncate)

In [21]:
df['身體檢查'].iloc[1] # ()裡面不一定會有紀錄

'[Physical Examination]\n\nVital signs:Date：20230720,Time：1629, BT：36.5℃, PR：88/min, RR：17/min, BP：141/72mmHg\n\nGeneral appearance:Normal health state, anxious\n\nConsciousness:GCS (Glasgow Coma Scale):E4M6V5\n\nHEENT: Head: trauma(-),facial swelling(-),puffy face(-)\n\nEye:conjunctiva: anemic(-), sclera:icteric(-)\n\nEars:otorrhea(-)\n\nNose:rhinorrhea(+)for 1 year,discharge(+)for 1 year off and on \n\nThroat:lip cyanosis(-),oral ulcer(-),injected(-),tonsil enlargement(-),tonsil pus coating(-)\n\nNeck:supple,thyroid enlargement(-),jugular vein engorgement(-),carotid bruit(-),lymphadenopathy(-)\n\nChest: Inspection:\n\naccessory muscle respiration(-),paradoxical movement(-)\n\nPalpation:tender point(-),palpable mass(-),tactile fremitus：symmetrical\n\nPercussion:abnormal dullness(-)\n\nAuscultation:clear breathing sound, rales(-),rhonchi(-),wheezing(-)\n\nHeart: Inspection:visible apical impulse\n\nPalpation:thrill(-),heave(-,palpable apical impulse at 5th intercostal space and left mi

In [22]:
# 建立文本
df['clincal_text'] = df.apply(build_text, axis=1)

In [23]:
# clincal_text字數
df['text_length'] = df['clincal_text'].apply(len)

In [24]:
df['text_length']

0    5890
1    4790
2    4353
3    5142
4    4723
5    4298
6    3646
7    6495
8    7043
9    4782
Name: text_length, dtype: int64

In [25]:
text_df = df['clincal_text']

In [27]:
text_df.to_excel('text_df.xlsx')

<p style="font-size:30; color:blue">文本</p>

In [23]:
PROMPT_TEMPLATE = f"""

你是一位「耳鼻喉科臨床醫師」兼「醫療病歷編碼專家」，負責從病歷中找出：  

1. 可能被遺漏的黴菌性鼻竇炎個案 
2. 是否具備黴菌性鼻竇炎特徵
3. 是否存在合併症   
4. 若資料不足 → 標記 Missing 並明確列出缺失項  

============================================================
第一部分：黴菌性鼻竇炎診斷條件（OR 規則）
============================================================

若以下任一出現，則判定為「疑似黴菌性鼻竇炎」（請提出"需補編 B49 或更精準 ICD"的建議）:

1. 臨床症狀（任一）
- 呼吸有臭味  
- 鼻出血 / 血水鼻涕  
- 單側鼻竇症狀  
- 臉部壓痛  
- 嗅覺喪失  
- 黑色壞死物、黑痂皮  

2. 影像學 CT/MRI（任一）
- 單側鼻竇病變  
- 鈣化點  
- Metallic density  
- High density lesion  
- Bone erosion  
- Double density sign  

3. 手術所見（任一）
- Fungal ball / Mycetoma  
- Cheesy / Caseous material  
- Clay-like material  
- 黑色或褐色碎屑  
- 壞死組織  

4. 病理 / 檢驗（任一）
- Fungus / fungal elements  
- Hyphae  
- Spores  
- Aspergillus  
- 真菌培養陽性  

============================================================
第二部分：合併症判定（僅限已判定「疑似黴菌性鼻竇炎」時啟動）
============================================================

若病例中任一出現 → 判定「有合併症」 → DRG = **41501**  
若皆無 → DRG = **41502**

A. 合併症 ICD-10 清單  
（感染、胸腔、頭頸、心血管、腎臟、代謝、GI 出血、其他，完整清單如原提示）

B. 檢驗異常  
- Na < 135 → 視為合併症（建議補編 E871）  
- Na > 145 → 視為合併症（建議補編 E870）  

============================================================
第三部分：Missing / Indeterminate 規則
============================================================

若下列任一資料缺失 → 全案標記「Missing / 無法判定」：  
- 無影像報告  
- 應有手術記錄但缺失  
- 無病理資料（若病例應有）  
- 症狀描述不足  
- 無 Na 檢驗值  
- 僅一般敘述，無可判定資訊  

不能因資料缺失直接判定「不是黴菌性鼻竇炎」。

需在輸出中列出：**缺失資料清單**，提示資訊組補資料。

============================================================
第四部分：輸出格式（固定格式）
============================================================

每一病例輸出一個表格，且去掉json的字符，生成格式如下：

Case ID:

觸發的關鍵字（具體引述):

特徵分類（1-5）:

黴菌性鼻竇炎判定: (Yes/No)

是否有合併症:

合併症 ICD:

建議補編 ICD:

建議 DRG:

最終建議:


若為 Missing，則在表格後加上：

缺失資料：  
- a.
- b.
- c.

============================================================
第五部分：內部邏輯（模型必須遵守）
============================================================

1. 忽略主診斷，只看全文實際內容  
2. 依 OR 規則偵測是否可能為黴菌性鼻竇炎  
3. 若符合 → 啟動合併症判定（ICD + Na）  
4. 若不符合 → 判定「不是黴菌性鼻竇炎」  
5. 若資訊不足 → Missing，列出缺失資料  
6. 若符合真菌特徵 → 建議補編 B49 或更精準 ICD（如 J32.x + B49)
7. 自動決定 DRG（41501 / 41502 / Missing）

"""

In [24]:
# 1. 定義單一任務的處理函數
def call_gpt_api(item):
    idx, row = item # 拆解傳入的元組
    
    messages = [
        {"role": "system", "content": PROMPT_TEMPLATE},
        {"role": "user", "content": f"病例數據：{row}\n請根據上面的病例與邏輯，給出診斷建議。"}
    ]
    
    try:
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0.1,
            top_p=0.9,
            max_tokens=800
        )
        content = response.choices[0].message.content
        return {"index": idx, "content": content, "status": "success"}
    except Exception as e:
        return {"index": idx, "content": str(e), "status": "error"}

In [25]:
# 2. 準備任務清單 (將 df 轉為 list)
tasks = list(enumerate(text_df)) 

k = 5 
results = []

with ThreadPoolExecutor(max_workers=k) as executor:
    results = list(executor.map(call_gpt_api, tasks))

results.sort(key=lambda x: x['index'])

for res in results:
    print(f"病患 {res['index'] + 1} 診斷結果：")
    print(res['content'])
    print("=" * 50)

病患 1 診斷結果：
Case ID:  
0000137913  

觸發的關鍵字（具體引述):  
- 病史中提到 "Chronic paranasal sinusitis at sphenoid, ethmoid and left maxillary sinuses"  
- CT 影像學報告顯示 "post operative change of the left maxillary sinus uncinate process" 和 "a 1.0 cm retention cyst of the left maxillary sinus"  
- 手術記錄中提到 "Open and cleared left sphenoid sinus"  

特徵分類（1-5）:  
1. 臨床症狀: 無明確符合黴菌性鼻竇炎的症狀  
2. 影像學 CT/MRI: 無明確符合黴菌性鼻竇炎的特徵  
3. 手術所見: 無明確符合黴菌性鼻竇炎的特徵  
4. 病理 / 檢驗: 病理報告 Pending，無法判定  
5. 其他: 無  

黴菌性鼻竇炎判定: No  

是否有合併症: No  

合併症 ICD: 無  

建議補編 ICD: 無  

建議 DRG: 41502  

最終建議:  
目前無法判定為黴菌性鼻竇炎，建議持續觀察並等待病理報告結果。  

缺失資料：  
- a. 病理報告（Pending）  
- b. 無明確黴菌性鼻竇炎相關症狀（如黑色壞死物、血水鼻涕等）  
- c. 無影像學特徵（如鈣化點、Bone erosion 等）  
- d. 無檢驗結果顯示真菌感染（如真菌培養陽性）  
- e. 無 Na 檢驗值
病患 2 診斷結果：
Case ID:  
0003139900  

觸發的關鍵字（具體引述):  
- 症狀: "Nasal obstruction and rhinorrhea off and on for 1 year"  
- 影像學: "increased soft tissue density at left maxillary sinus"  
- 手術所見: "Removed left fungal ball"  

特徵分類（1-5）:  
1. 臨床症狀  
2. 影像學 CT/MRI  
3. 手術所見  